# 1. Import Required Libraries

This notebook focuses on training and evaluating multiple regression models for predicting the estimated repair cost.

The required libraries include:

- Data manipulation
- Model training
- Model evaluation
- Model persistence
- Warning suppression

In [6]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Evaluation Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 2. Load Dataset

The cleaned dataset generated during preprocessing is loaded for model training.

This dataset contains the selected features after preprocessing and feature selection.

In [7]:
df = pd.read_csv("../data/processed/processed_data.csv")


# 3. Separate Features and Target

The dataset is divided into:

- Features (X)
- Target Variable (y)

Target:
- estimated_repair_cost

In [8]:
X = df.drop(columns=["estimated_repair_cost"])
y = df["estimated_repair_cost"]

print("Features :", X.shape)
print("Target   :", y.shape)

Features : (24042, 8)
Target   : (24042,)


# 4. Train-Test Split

The dataset is divided into training and testing subsets.

- Training Set : 80%
- Testing Set  : 20%

The same random state used during preprocessing is maintained for reproducibility.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (19233, 8)
X_test  : (4809, 8)
y_train : (19233,)
y_test  : (4809,)


# 5. Load Preprocessing Pipeline

The preprocessing pipeline saved during the preprocessing stage is loaded.

This ensures that the exact same preprocessing steps used during training are also applied during inference.

In [10]:
preprocessor = joblib.load("./model/preprocessor.pkl")

print("Preprocessor Loaded Successfully.")

Preprocessor Loaded Successfully.


# 6. Transform Dataset

The preprocessing pipeline is fitted on the training data and then applied to both the training and testing datasets.

This ensures consistency between model training and prediction.

In [11]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape :", X_train_processed.shape)
print("Processed Testing Shape  :", X_test_processed.shape)

Processed Training Shape : (19233, 17)
Processed Testing Shape  : (4809, 17)



# 7. Create a Model Evaluation Function

Instead of writing repetitive code for each regression model, a reusable evaluation function is created.

The function performs the following tasks:

- Train the model
- Predict on training data
- Predict on testing data
- Calculate evaluation metrics
- Return the model performance

The following metrics are used:

- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- R² Score

In [12]:
def evaluate_model(model, X_train, X_test, y_train, y_test):

    # Train Model
    model.fit(X_train, y_train)

    # Predictions
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Training Metrics
    train_mae = mean_absolute_error(y_train, train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    train_r2 = r2_score(y_train, train_pred)

    # Testing Metrics
    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_r2 = r2_score(y_test, test_pred)

    return {
        "Model": model.__class__.__name__,
        "Train MAE": round(train_mae, 2),
        "Test MAE": round(test_mae, 2),
        "Train RMSE": round(train_rmse, 2),
        "Test RMSE": round(test_rmse, 2),
        "Train R2": round(train_r2, 4),
        "Test R2": round(test_r2, 4)
    }

# 8. Train Multiple Regression Models

Six regression algorithms are trained and evaluated.

Models included:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

Each model is evaluated using the same training and testing datasets to ensure a fair comparison.

In [13]:
models = [

    LinearRegression(),

    Ridge(random_state=42),

    Lasso(random_state=42),

    DecisionTreeRegressor(random_state=42),

    RandomForestRegressor(random_state=42),

    GradientBoostingRegressor(random_state=42)

]

# 9. Model Evaluation

Each regression model is trained using the evaluation function.

The performance metrics are collected into a comparison table to identify the best-performing model.

In [14]:
results = []

for model in models:

    result = evaluate_model(
        model,
        X_train_processed,
        X_test_processed,
        y_train,
        y_test
    )

    results.append(result)

In [15]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    by="Test R2",
    ascending=False,
    inplace=True
)

results_df.reset_index(drop=True, inplace=True)

results_df

,Model,Train MAE,Test MAE,Train RMSE,Test RMSE,Train R2,Test R2
0,LinearRegression,139.46,137.69,415.94,406.33,0.9291,0.9342
1,Ridge,139.69,137.93,415.94,406.38,0.9291,0.9342
2,Lasso,137.71,136.06,416.09,406.84,0.9291,0.9341
3,GradientBoostingRegressor,128.35,134.70,394.20,410.89,0.9363,0.9328
4,RandomForestRegressor,51.82,139.36,167.33,439.81,0.9885,0.9230
5,DecisionTreeRegressor,0.00,190.62,0.00,618.52,1.0000,0.8476


# Baseline Model Evaluation

Multiple regression models were trained and evaluated using the test dataset. The performance of each model was compared using MAE, RMSE, and R² Score.

## Linear Regression
- Achieved the highest Test R² Score (0.9342).
- Train and Test performance were very similar, indicating good generalization.
- No signs of overfitting.

## Ridge Regression
- Produced almost identical results to Linear Regression.
- Regularization did not improve the model performance.

## Lasso Regression
- Performance was also very close to Linear Regression.
- Feature selection through Lasso did not provide additional benefit.

## Gradient Boosting Regressor
- Achieved a strong Test R² Score (0.9328).
- Slightly lower performance than Linear Regression.
- Mild overfitting was observed.

## Random Forest Regressor
- Achieved a very high Train R² Score but lower Test R² Score.
- This indicates overfitting on the training data.

## Decision Tree Regressor
- Perfectly fitted the training data but performed poorly on the test data.
- Severe overfitting was observed.

## Model Selection

Based on the baseline evaluation, **Linear Regression** was selected as the best baseline model because it achieved:

- Highest Test R² Score (0.9342)
- Lowest prediction error
- Excellent generalization
- Simple and interpretable model with no overfitting

Although Gradient Boosting also performed well, Linear Regression provided slightly better performance with much lower model complexity.

# 10. Cross Validation

Model performance based on a single train-test split may vary depending on the data partition.

To obtain a more reliable estimate of model performance, K-Fold Cross Validation is performed.

In this project:

- 8-Fold Cross Validation is used.
- The R² score is selected as the evaluation metric.
- Mean and Standard Deviation of the R² scores are calculated.

A model with a higher mean R² and lower standard deviation is considered more stable and reliable.

In [11]:
from sklearn.model_selection import cross_val_score
cv_models = {

    "Linear Regression": LinearRegression(),

    "Gradient Boosting": GradientBoostingRegressor(random_state=42),

    "Random Forest": RandomForestRegressor(random_state=42),

    "Decision Tree" : DecisionTreeRegressor(random_state=42),

    "Lasso" : Lasso(random_state=42),

    "Ridge" : Ridge(random_state=42)
    

}
cv_results = []

for name, model in cv_models.items():

    scores = cross_val_score(
        model,
        X_train_processed,
        y_train,
        cv=8,
        scoring="r2",
        n_jobs=-1,
        verbose=2
    )

    cv_results.append({
        "Model": name,
        "Mean R2": round(scores.mean(),4),
        "Std Dev": round(scores.std(),4)
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df.sort_values(
    by="Mean R2",
    ascending=False,
    inplace=True
)

cv_results_df.reset_index(drop=True,inplace=True)

cv_results_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   8 | elapsed:    2.6s remaining:    1.5s
[Parallel(n_jobs=-1)]: Done   8 out of   8 | elapsed:    2.6s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   8 | elapsed:    5.2s remaining:    3.1s
[Parallel(n_jobs=-1)]: Done   8 out of   8 | elapsed:    5.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   8 | elapsed:    2.8s remaining:    1.7s
[Parallel(n_jobs=-1)]: Done   8 out of   8 | elapsed:    2.9s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   8 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done   8 out of   8 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parall

,Model,Mean R2,Std Dev
0,Linear Regression,0.9290,0.0047
1,Lasso,0.9290,0.0047
2,Ridge,0.9290,0.0047
3,Gradient Boosting,0.9261,0.0046
4,Random Forest,0.9170,0.0048
5,Decision Tree,0.8507,0.0091


# Cross Validation

To evaluate the stability and generalization of the models, **8-Fold Cross Validation** was performed using the **R² Score**.

| Model | Mean CV R² |
|--------|-----------:|
| Linear Regression | **0.9290** |
| Ridge Regression | **0.9290** |
| Lasso Regression | **0.9290** |
| Gradient Boosting | 0.9261 |
| Random Forest | 0.9170 |
| Decision Tree | 0.8507 |

## Conclusion

Linear Regression, Ridge Regression, and Lasso Regression achieved the highest average CV R² Score (0.9290), indicating consistent and stable performance across all 8 folds.

Therefore, **Linear Regression** was selected as the best baseline model due to its high performance, excellent generalization, and simple architecture.

# 11. Hyperparameter Tuning

Hyperparameter tuning is performed to optimize the performance of the best-performing tree-based models.

RandomizedSearchCV is used because it efficiently explores the hyperparameter space while reducing computational cost.

The objective is to identify the parameter combination that produces the best cross-validation performance.

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
rf_params = {

    "n_estimators":[100,200,300,400],

    "max_depth":[None,2,3,4,5,6,7,10,20,30],

    "min_samples_split":[2,3,5,10],

    "min_samples_leaf":[1,2,4,6,8]

}

In [ ]:
rf = RandomForestRegressor(random_state=42)

rf_search = GridSearchCV(

    estimator=rf,

    param_grid=rf_params,



    cv=8,

    scoring="r2",


    n_jobs=-1,
    verbose=2

)

rf_search.fit(X_train_processed,y_train)

print("Best Parameters")
print(rf_search.best_params_)

print()

print("Best CV Score")
print(rf_search.best_score_)

In [ ]:
gb_params={

    "n_estimators":[100,200,300,400,500],

    "learning_rate":[0.01,0.05,0.1],

    "max_depth":[3,4,5],

    "subsample":[0.8,0.9,1.0]

}

In [ ]:
gb = GradientBoostingRegressor(random_state=42)

gb_search = GridSearchCV(

    estimator=gb,

    param_grid=gb_params,

    cv=8,

    scoring="r2",

    n_jobs=-1,
    verbose=2

)

gb_search.fit(X_train_processed,y_train)

print("Best Parameters")
print(gb_search.best_params_)

print()

print("Best CV Score")
print(gb_search.best_score_)

In [ ]:
tuned_models = [

    {
        "Model":"Random Forest",
        "Best CV Score":rf_search.best_score_
    },

    {
        "Model":"Gradient Boosting",
        "Best CV Score":gb_search.best_score_
    }

]

pd.DataFrame(tuned_models)

<h2>📊 Before Hyperparameter Tuning</h2>

| Model             |      CV R² |
| ----------------- | ---------: |
| Linear Regression | **0.9290** |
| Gradient Boosting |     0.9261 |
| Random Forest     |     0.9170 |


<h2>📊 After Hyperparameter Tuning</h2>

| Model                         | Best CV R² | Improvement |
| ----------------------------- | ---------: | ----------: |
| **Linear Regression**         | **0.9290** |           — |
| **Random Forest (Tuned)**     | **0.9286** |  ⬆️ +0.0116 |
| **Gradient Boosting (Tuned)** | **0.9280** |  ⬆️ +0.0019 |


In [16]:

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# Train Model
lr=LinearRegression()

lr.fit(X_train_processed, y_train)

# Predictions
train_pred = lr.predict(X_train_processed)
test_pred = lr.predict(X_test_processed)

# Adjusted R2 Function
def adjusted_r2(r2, n, p):
    return 1 - ((1-r2)*(n-1)/(n-p-1))

# Train Metrics
train_r2 = r2_score(y_train, train_pred)
train_adj_r2 = adjusted_r2(train_r2, X_train.shape[0], X_train.shape[1])

# Test Metrics
test_r2 = r2_score(y_test, test_pred)
test_adj_r2 = adjusted_r2(test_r2, X_test.shape[0], X_test.shape[1])

print("="*60)
print("Linear Regression Finalised Model)")
print("="*60)

print(f"Train MAE  : {mean_absolute_error(y_train, train_pred):.2f}")
print(f"Test MAE   : {mean_absolute_error(y_test, test_pred):.2f}")

print(f"Train RMSE : {np.sqrt(mean_squared_error(y_train, train_pred)):.2f}")
print(f"Test RMSE  : {np.sqrt(mean_squared_error(y_test, test_pred)):.2f}")

print(f"Train R²   : {train_r2:.4f}")
print(f"Test R²    : {test_r2:.4f}")

print(f"Train Adjusted R² : {train_adj_r2:.4f}")
print(f"Test Adjusted R²  : {test_adj_r2:.4f}")

Linear Regression Finalised Model)
Train MAE  : 139.46
Test MAE   : 137.69
Train RMSE : 415.94
Test RMSE  : 406.33
Train R²   : 0.9291
Test R²    : 0.9342
Train Adjusted R² : 0.9291
Test Adjusted R²  : 0.9341


## Hyperparameter Tuning

Hyperparameter tuning was considered for the tree-based models. However, the baseline evaluation and 8-Fold Cross Validation showed that Linear Regression already achieved the highest and most stable performance.

Therefore, additional tuning was not performed, as it was unlikely to provide a meaningful improvement over the selected model.

Final Selected Model: **Linear Regression**

# Final Conclusion

Six regression models were evaluated for predicting machine maintenance cost.

Based on the baseline evaluation and 8-Fold Cross Validation, **Linear Regression** consistently achieved the best performance.

### Final Model Performance

- Test R² Score: **0.9342**
- Mean CV R² (8-Fold): **0.9290**

Linear Regression was selected as the final model because it provided:

- Highest predictive performance
- Excellent generalization
- No overfitting
- Simple and interpretable architecture
- Fast training and prediction

Therefore, **Linear Regression** was chosen as the final model and saved for deployment.

In [17]:
from sklearn.linear_model import LinearRegression
import joblib
import os

model = LinearRegression()
model.fit(X_train_processed, y_train)

os.makedirs("model", exist_ok=True)
joblib.dump(model, "model/LinearRegression.joblib")
print("Model Saved Successfully")

Model Saved Successfully
